# Linear Equations

Every quantum computation in this curriculum is, underneath, a pile of linear
equations being solved at once. Before a matrix can mean anything, the equations it
came from have to. So this notebook starts at `4x + 5 = 17` and ends with a grid of
numbers -- the object the next three notebooks are about.

Nothing here needs quantum mechanics, an AWS account, or anything past high-school
algebra. Work each answer on paper first and let NumPy be the marker. That order is
deliberate: the point of this module is that *you* can do the arithmetic, not that
NumPy can.

**Objectives:**
- Recognize a linear equation, and name what disqualifies the ones that are not
- Solve a two-unknown system by substitution and by elimination
- Solve a three-unknown system by elimination
- Tell the three possible outcomes apart: one solution, none, infinitely many
- Strip the unknowns off a system and read the grid of coefficients left behind

**Reference:** See [../GUIDE.md](../GUIDE.md).

<!-- browser-runnable -->

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. What makes an equation linear

An equation is **linear** when every unknown in it appears alone: multiplied by a
plain number, raised to the first power, and added to the other terms. That is the
whole rule. Nothing else is allowed to happen to an unknown.

These are linear:

- $4x + 5 = 17$
- $2a - 3b + c = 0$
- $y = 4 - 2x$

These are not, and each one fails for a different reason:

- $x^2 + y = 4$ -- an unknown is raised to a power
- $xy = 6$ -- two unknowns are multiplied together
- $\sin(x) + y = 1$ -- an unknown sits inside a function
- $1/x + y = 3$ -- an unknown sits in a denominator

The name comes from the picture: an equation in two unknowns that obeys the rule
draws a straight line, and one that breaks it draws a curve.

The rule also buys a kind of predictability, though it has to be stated carefully.
In an expression with no constant term added on -- $2x + 3y$, say -- doubling both
inputs doubles the value exactly. Add a constant, as $4x + 5$ does, and the doubling
no longer lands on the nose: at $x = 1$ the value is 9, and at $x = 2$ it is 13
rather than 18. That expression is still linear, because linearity is a rule about
how the unknowns appear, not a promise about doubling. The cell below shows all three
cases, including the curved one, which fails the rule outright.

In [ ]:
def homogeneous_expr(x, y):
    """2x + 3y -- every unknown stands alone and nothing is added on top."""
    return 2 * x + 3 * y


def curved_expr(x, y):
    """x squared + 3y -- one unknown is raised to a power, so this is not linear."""
    return x ** 2 + 3 * y


def offset_expr(x):
    """4x + 5 -- linear: the unknown stands alone and 5 is only a constant."""
    return 4 * x + 5


print("homogeneous_expr(1, 1) =", homogeneous_expr(1, 1),
      "  homogeneous_expr(2, 2) =", homogeneous_expr(2, 2))
print("  doubling the inputs doubled it:",
      homogeneous_expr(2, 2) == 2 * homogeneous_expr(1, 1))

print("curved_expr(1, 1) =", curved_expr(1, 1),
      "  curved_expr(2, 2) =", curved_expr(2, 2))
print("  doubling did not double it, and it is not linear:",
      curved_expr(2, 2) == 2 * curved_expr(1, 1))

print("offset_expr(1) =", offset_expr(1), "  offset_expr(2) =", offset_expr(2))
print("  doubling did not double it either, yet it IS linear:",
      offset_expr(2) == 2 * offset_expr(1))

Now the notation, which is only that rule written compactly. A linear equation in
`n` unknowns is

$$
a_1 x_1 + a_2 x_2 + \cdots + a_n x_n = b
$$

The $x_i$ are the **unknowns**, the $a_i$ are the **coefficients** (known numbers),
and $b$ is the **constant term** on the right. Each unknown appears exactly once, to
the first power, multiplied by its coefficient. If your equation can be rearranged
into that shape, it is linear.

One convention worth fixing now, because it causes real confusion later: mathematics
counts from one, so the first coefficient is $a_1$. NumPy counts from zero, so the
same number is `a[0]`. Both appear throughout this module, and every time the two
meet, the text says which one it means.

## 2. One unknown

With a single unknown, solving means peeling away everything around it one inverse
operation at a time, doing the same thing to both sides so the equation stays true.

Take $4x + 5 = 17$. Subtract 5 from both sides and it becomes $4x = 12$. Divide both
sides by 4 and it becomes $x = 3$.

That is the entire method, and it never changes. Systems with more unknowns are this
same move applied after some bookkeeping that gets the other unknowns out of the way.

**Always check by substituting.** Put the answer back into the original equation and
evaluate both sides. It costs one line and it catches every arithmetic slip you will
ever make. This is the one habit from this notebook that survives all the way into
hardware runs.

In [ ]:
one_unknown_x = (17 - 5) / 4
print("x =", one_unknown_x)
print("substituting back, 4x + 5 =", 4 * one_unknown_x + 5, "-- and the equation asks for 17")

## 3. Two unknowns, by substitution

One equation in two unknowns pins nothing down. $x + y = 7$ is satisfied by
$(0, 7)$, by $(1, 6)$, by $(2.5, 4.5)$ -- a whole line of pairs. To pin a pair down
you need a second, genuinely different equation. A **system** of equations is a set
of equations that must all hold at once, and a **solution** is one assignment of
values that satisfies every one of them.

**Substitution** is the method to reach for when one equation already tells you what
an unknown equals, or can be rearranged until it does. Solve that equation for one
unknown, push the result into the other equation, and you are back to the
one-unknown case from Section 2.

$$
\begin{aligned}
x + y &= 7 \\
y &= x + 1
\end{aligned}
$$

The second equation already says what $y$ is. Substituting it into the first gives
$x + (x + 1) = 7$, so $2x + 1 = 7$, so $2x = 6$ and $x = 3$. Then the second equation
returns $y = 4$.

In [ ]:
sub_x = 3
sub_y = 4
print("x + y =", sub_x + sub_y, "-- the first equation asks for 7")
print("y     =", sub_y, "-- the second asks for x + 1 =", sub_x + 1)

## 4. Two unknowns, by elimination

Substitution needs an equation that rearranges cleanly. Often neither one does.
**Elimination** does not care: scale the equations until one unknown carries equal
and opposite coefficients, then add the two equations together and that unknown
vanishes.

$$
\begin{aligned}
2x + y &= 11 \\
x - y &= 1
\end{aligned}
$$

The $y$ terms are already $+y$ and $-y$, so adding the two equations straight away
gives $3x = 12$, and $x = 4$. Putting that back into the second equation leaves $y$
as the only unknown there, and $y = 3$.

If the coefficients had not lined up, you would multiply one or both equations by a
number first. Multiplying an entire equation by a non-zero number does not change
which pairs satisfy it, and that freedom is the whole of elimination. It is also,
though it will not look like it yet, the row operation that drives every matrix
method in linear algebra.

In [ ]:
elim_x = 4
elim_y = 3
print("2x + y =", 2 * elim_x + elim_y, "-- the first equation asks for 11")
print("x - y  =", elim_x - elim_y, "-- the second asks for 1")

## 5. Three unknowns

Elimination scales. With three equations and three unknowns, combine them in pairs to
knock out one unknown at a time until a single unknown is left, then walk the values
back up.

$$
\begin{aligned}
x + y + z &= 9 \\
x - y + z &= 3 \\
2x + y - z &= 3
\end{aligned}
$$

Subtract the second equation from the first: the $x$ and $z$ terms cancel and what is
left is $2y = 6$, so $y = 3$. Add the first and third: the $z$ terms cancel and what
is left is $3x + 2y = 12$; with $y = 3$ that is $3x = 6$, so $x = 2$. The first
equation then returns $z = 4$.

Two things to notice. The bookkeeping grew but the moves did not -- scale, add,
substitute back. And the coefficients did all of the work; the letters $x$, $y$ and
$z$ were carried along unchanged from the first line to the last. Hold that thought.

In [ ]:
three_x, three_y, three_z = 2, 3, 4
print("x + y + z  =", three_x + three_y + three_z, "-- asks for 9")
print("x - y + z  =", three_x - three_y + three_z, "-- asks for 3")
print("2x + y - z =", 2 * three_x + three_y - three_z, "-- asks for 3")

## 6. Exactly one solution, none, or infinitely many

Two linear equations in two unknowns are two straight lines drawn on the same axes,
and two straight lines can relate to each other in only three ways. That is why there
are exactly three outcomes, with nothing else possible.

**They cross at one point -- exactly one solution.** The lines have different slopes,
so they meet once and only once. Every system solved so far has been this case.

**They are parallel and distinct -- no solution.** Same slope, different intercept.
Written out, the two equations ask the same combination of unknowns to equal two
different numbers, which nothing can do. Such a system is called *inconsistent*.

**They are the same line -- infinitely many solutions.** The second equation is the
first one scaled by a number, so it says nothing new. Every point on the line is a
solution, and the system is one honest equation dressed as two.

The test is mechanical. Scale one equation until its coefficients match the other's.
If the constant terms then match too, the lines coincide; if they do not, the lines
are parallel; and if the coefficients cannot be made to match at all, the lines
cross.

In [ ]:
print("parallel and distinct:   x + y = 3   and   x + y = 5")
print("  subtracting one from the other leaves 0 = 2, which no pair can satisfy")
print()
print("the same line twice:     x + y = 3   and   2x + 2y = 6")
for cand_x in [0, 1, 2, 3]:
    cand_y = 3 - cand_x
    print(f"  (x, y) = ({cand_x}, {cand_y}) satisfies both:"
          f"  x + y = {cand_x + cand_y},  2x + 2y = {2 * cand_x + 2 * cand_y}")

## 7. Strip the unknowns and a grid is what is left

Look again at the three-unknown system from Section 5:

$$
\begin{aligned}
x + y + z &= 9 \\
x - y + z &= 3 \\
2x + y - z &= 3
\end{aligned}
$$

The letters never changed. Every step of the elimination moved coefficients around
and carried $x$, $y$ and $z$ along as labels. So write down only what actually moved:

```
  1   1   1   |   9
  1  -1   1   |   3
  2   1  -1   |   3
```

The block on the left is a **matrix** -- a rectangular grid of numbers with a shape,
here three rows by three columns. Row `i` is equation `i`; column `j` holds every
coefficient of the `j`-th unknown. The column on the right is the list of constant
terms. Storing a system this way is not a notational nicety: it is what lets a
computer solve ten thousand equations without ever seeing a letter.

That grid is the subject of the next three notebooks -- how to add them, how to
multiply them, how to flip them around.

One more thing hides here. Checking a solution means substituting it back: take a row
of coefficients, multiply each one by the matching unknown's value, and add the
results up. Do that for every row and you have turned a grid and a column of values
into a new column of numbers. That operation has a name, it is the single most used
operation in quantum computing, and notebook 03 is entirely about it. You have been
doing it by hand all along.

In [ ]:
demo_grid = np.array([
    [1,  1,  1],
    [1, -1,  1],
    [2,  1, -1],
])
demo_rhs = np.array([9, 3, 3])
demo_sol = np.array([2, 3, 4])

print("coefficient grid:")
print(demo_grid)
print("shape (rows, columns):", demo_grid.shape)
print("constant terms:", demo_rhs)

demo_substituted = np.sum(demo_grid * demo_sol, axis=1)
print("substituting the solution row by row:", demo_substituted)
print("which reproduces the constant terms:", np.array_equal(demo_substituted, demo_rhs))

## 8. Notation cheat sheet

| Idea | Written as | What it means here |
|---|---|---|
| Linear equation | `a1*x1 + a2*x2 + ... + an*xn = b` | every unknown alone, to the first power |
| System | several such equations at once | all of them must hold together |
| Solution | one value per unknown | substituting it back reproduces every constant term |
| Coefficient matrix | the grid of the `a` values | rows are equations, columns are unknowns |
| Constant column | the `b` values | one number per equation |

Three habits worth carrying forward:

1. **Substitute back, every time.** It is the only check that does not depend on
   trusting the method you just used.
2. **Reach for elimination when nothing rearranges cleanly.** Substitution is faster
   when an equation is already solved for an unknown; elimination never stalls.
3. **Read the coefficients, not the letters.** The letters are labels. Everything
   that happens, happens to the numbers.

## 9. Exercises

Practice comes in two forms here. First an unlimited drill generator, then six graded
exercises, each with tiered hints and a check cell.

The drill hands you a coefficient grid and a column of values and asks for the
substitution -- the row-by-row multiply-and-add from Section 7. `level=1` gives a
two-by-two grid with small non-negative numbers. `level=2` is larger and signed:
entries may be negative, and the grid may be three-by-three rather than two-by-two.
Every problem is reproducible from its seed, so one you got wrong can be worked again
later.

Work each problem on paper before you check it. `check` reports which entries are off
and by how much; `reveal` is the only thing that shows a worked answer.

In [ ]:
from lib.linalg_drills import drill

practice = drill("matvec", level=1, seed=21)
practice.show()

# This one works out to [18, 16]. Hand an answer over as a list:
#     practice.check([18, 16])
# Stuck? practice.reveal() shows the worked answer.
# Omit the seed for a fresh problem -- the drawn seed is printed so you can
# return to it later:  drill("matvec", level=2)

### Exercise 1 -- Linear or not

Six equations. Decide, for each one, whether it is linear.

$$
\begin{aligned}
\text{(a)} \quad & 3x - 7y = 12 \\
\text{(b)} \quad & x^2 + y = 4 \\
\text{(c)} \quad & y = 4 - 2x \\
\text{(d)} \quad & xy = 6 \\
\text{(e)} \quad & \sin(x) + y = 1 \\
\text{(f)} \quad & 5x + 2y - z = 0
\end{aligned}
$$

Define `ex1_linear`: a list of six booleans in the order (a) through (f), where
`True` means that equation is linear.

<details><summary>Hint 1 -- nudge</summary>

Section 1 gave one rule and four ways to break it. Read each equation asking a single
question about every unknown in it: does that unknown appear on its own, multiplied
by a plain number and added to the rest? Rearranging an equation never changes the
answer, and the number of unknowns never matters.

</details>
<details><summary>Hint 2 -- approach</summary>

Go one equation at a time and write down the reason before the verdict. You are
looking for four disqualifiers: a power other than one, two unknowns multiplied
together, an unknown inside a function, and an unknown in a denominator. If none of
them is present, the equation is linear.

</details>

In [ ]:
# Exercise 1: Decide which of the six equations are linear.
# Define: ex1_linear -- six booleans, in the order (a) through (f).

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    verdicts = [bool(v) for v in ex1_linear]
    assert len(verdicts) == 6, (
        "one verdict per equation -- six in all, in the order (a) through (f)"
    )
    expected_linear = [True, False, True, False, False, True]
    off = [chr(ord("a") + i) for i in range(6) if verdicts[i] != expected_linear[i]]
    assert not off, (
        "look again at equation(s) " + ", ".join(off) + " -- an unknown may only be "
        "multiplied by a plain number and added; a power, a product of two unknowns, "
        "a function, or a denominator disqualifies it"
    )

### Exercise 2 -- Solve by substitution

$$
\begin{aligned}
y &= 3x - 1 \\
2x + y &= 9
\end{aligned}
$$

Solve this system by substitution and record the answer.

Define `ex2_x` and `ex2_y` -- the two values, each a single number, not a list and
not an array.

The check substitutes your pair back into both equations rather than comparing it
against a stored answer, so any route that gets there is accepted. Work it on paper
first.

<details><summary>Hint 1 -- nudge</summary>

Section 3 said substitution is the method to reach for when one equation already
tells you what an unknown equals. Read the two equations and notice that one of them
is already in exactly that form, with no rearranging needed.

</details>
<details><summary>Hint 2 -- approach</summary>

Replace `y` in the second equation with the expression the first equation gives for
it. That leaves one equation in `x` alone, which is the Section 2 case: collect the
`x` terms, move the number to the other side, divide. Feed `x` back into the first
equation to recover `y`, then substitute the pair into both equations before you
trust it.

</details>

In [ ]:
# Exercise 2: Solve the two-unknown system by substitution.
# Define: ex2_x, ex2_y -- the solution values, as plain numbers.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert np.asarray(ex2_x).shape == () and np.asarray(ex2_y).shape == (), (
        "give each unknown as a single number, not a list and not an array"
    )
    x2, y2 = float(ex2_x), float(ex2_y)
    assert np.isclose(y2, 3 * x2 - 1), (
        "substitute your pair into the first equation -- the two sides disagree, so "
        "re-check the value you found for y"
    )
    assert np.isclose(2 * x2 + y2, 9), (
        "substitute your pair into the second equation -- its left side does not "
        "come out to the number on the right"
    )

### Exercise 3 -- Solve by elimination

$$
\begin{aligned}
4x + 3y &= 25 \\
2x - 3y &= -1
\end{aligned}
$$

Neither equation is arranged for substitution, so eliminate instead.

Define `ex3_x` and `ex3_y` -- the two values, each a single number.

<details><summary>Hint 1 -- nudge</summary>

Section 4 said to look for an unknown whose coefficients are already equal and
opposite. Compare the two numbers standing in front of `y` before you multiply
anything by anything.

</details>
<details><summary>Hint 2 -- approach</summary>

Add the two equations together term by term and one unknown disappears. Solve the
one-unknown equation that is left, then put that value into either original equation
to recover the other unknown. Finish by substituting the pair into both equations,
minus signs included.

</details>

In [ ]:
# Exercise 3: Solve the two-unknown system by elimination.
# Define: ex3_x, ex3_y -- the solution values, as plain numbers.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert np.asarray(ex3_x).shape == () and np.asarray(ex3_y).shape == (), (
        "give each unknown as a single number, not a list and not an array"
    )
    x3, y3 = float(ex3_x), float(ex3_y)
    assert np.isclose(4 * x3 + 3 * y3, 25), (
        "substitute your pair into the first equation -- its left side does not come "
        "out to the number on the right"
    )
    assert np.isclose(2 * x3 - 3 * y3, -1), (
        "substitute your pair into the second equation -- watch the minus sign in "
        "front of the y term and the minus sign on the constant"
    )

### Exercise 4 -- No solution, or infinitely many

Neither of these systems has exactly one solution. One of them has none at all, and
the other has infinitely many.

System P:

$$
\begin{aligned}
x + 2y &= 5 \\
2x + 4y &= 7
\end{aligned}
$$

System Q:

$$
\begin{aligned}
3x - y &= 4 \\
6x - 2y &= 8
\end{aligned}
$$

Define `ex4_no_solution` and `ex4_infinite` -- each the string `"P"` or `"Q"`, naming
the system that behaves that way. Do not try to solve either one.

<details><summary>Hint 1 -- nudge</summary>

Section 6 turned this into a picture: two lines that never meet, or two lines that
are secretly the same line. Take each system as a pair of lines and ask whether its
second equation says anything the first one did not.

</details>
<details><summary>Hint 2 -- approach</summary>

In each system, scale the first equation until its `x` coefficient matches the second
equation's, then compare the two equations term by term. If everything matches, the
constant term included, the two equations are one equation. If the left sides match
but the constants do not, the system is asking for something impossible.

</details>

In [ ]:
# Exercise 4: Say which system has no solution and which has infinitely many.
# Define: ex4_no_solution, ex4_infinite -- each the string "P" or "Q".

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    systems = {
        "P": ((1, 2, 5), (2, 4, 7)),
        "Q": ((3, -1, 4), (6, -2, 8)),
    }
    none_label = str(ex4_no_solution).strip().upper()
    many_label = str(ex4_infinite).strip().upper()
    assert {none_label, many_label} == {"P", "Q"}, (
        'name one system "P" and the other "Q" -- the two behave differently, so the '
        "same label cannot answer both"
    )
    (a1, b1, c1), (a2, b2, c2) = systems[none_label]
    assert np.isclose(a1 * b2, a2 * b1) and not np.isclose(a1 * c2, a2 * c1), (
        "the no-solution system is the one whose two left sides are multiples of each "
        "other while its two constant terms are not -- scale and compare again"
    )
    (d1, e1, f1), (d2, e2, f2) = systems[many_label]
    assert np.isclose(d1 * e2, d2 * e1) and np.isclose(d1 * f2, d2 * f1), (
        "the infinitely-many system is the one whose second equation is the first one "
        "scaled, constant term included"
    )

### Exercise 5 -- Three unknowns

$$
\begin{aligned}
x + y + z &= 6 \\
2x - y + z &= 3 \\
x + 2y - z &= 2
\end{aligned}
$$

Solve this system. Elimination is the method Section 5 used, but any route that lands
on values satisfying all three equations counts.

Define `ex5_x`, `ex5_y` and `ex5_z` -- each a single number.

<details><summary>Hint 1 -- nudge</summary>

Three equations, three unknowns. The plan from Section 5 was to combine equations in
pairs so that one unknown disappears, leaving a smaller system you already know how
to solve. Look down the columns for the unknown whose coefficients cancel most
easily.

</details>
<details><summary>Hint 2 -- approach</summary>

The `z` coefficients cancel when you add the right pairs: adding the first and third
equations leaves $2x + 3y = 8$, and adding the second and third leaves $3x + y = 5$.
That reduced pair is not ready for straight addition -- neither unknown carries equal
and opposite coefficients across the two equations -- so scale one of them first, the
way Section 4 described, until one unknown cancels. Solve for the two unknowns,
recover the third from any one of the original equations, then substitute all three
values into all three equations.

</details>

In [ ]:
# Exercise 5: Solve the three-unknown system.
# Define: ex5_x, ex5_y, ex5_z -- the solution values, as plain numbers.

# TODO: your code here

In [ ]:
# Check Exercise 5 -- run after your attempt.
from lib.grading import check

with check("Exercise 5"):
    assert (np.asarray(ex5_x).shape == () and np.asarray(ex5_y).shape == ()
            and np.asarray(ex5_z).shape == ()), (
        "give each unknown as a single number, not a list and not an array"
    )
    x5, y5, z5 = float(ex5_x), float(ex5_y), float(ex5_z)
    assert np.isclose(x5 + y5 + z5, 6), (
        "substitute your three values into the first equation -- they do not add up "
        "to the number on the right"
    )
    assert np.isclose(2 * x5 - y5 + z5, 3), (
        "substitute into the second equation -- check the coefficient 2 on x and the "
        "minus sign on y"
    )
    assert np.isclose(x5 + 2 * y5 - z5, 2), (
        "substitute into the third equation -- check the coefficient 2 on y and the "
        "minus sign on z"
    )

### Exercise 6 -- Build the grid, then substitute with NumPy

Section 7 stripped the unknowns off a system and left a grid of coefficients beside a
column of constants. Do that for the Exercise 5 system, then let NumPy perform the
substitution check you have been doing by hand.

$$
\begin{aligned}
x + y + z &= 6 \\
2x - y + z &= 3 \\
x + 2y - z &= 2
\end{aligned}
$$

Define four things:

- `ex6_coeffs` -- the coefficient matrix, one row per equation in the order written,
  columns in the order `x`, `y`, `z`
- `ex6_rhs` -- the three constant terms, as a flat 3-entry array
- `ex6_sol` -- your Exercise 5 answer as a flat 3-entry array, in the order `x`, `y`,
  `z`
- `ex6_lhs` -- the three left-side values you get by substituting `ex6_sol` into each
  row

<details><summary>Hint 1 -- nudge</summary>

Reading the matrix off the page is bookkeeping, not arithmetic: every coefficient
written down goes into a slot, and a coefficient of 1 or -1 is still a coefficient
even when the digit is not printed. Substituting means what it has always meant --
multiply each coefficient in a row by the matching value, then add the row up.

</details>
<details><summary>Hint 2 -- approach</summary>

Build the matrix with `np.array` from a list of three lists, one list per equation,
and build the other two as flat 3-entry arrays. For the substitution, multiplying the
matrix by the solution array with `*` pairs each coefficient with the value sitting
in its own column; summing along `axis=1` then adds each row up.

</details>

In [ ]:
# Exercise 6: Put the Exercise 5 system into a matrix and verify it with NumPy.
# Define: ex6_coeffs, ex6_rhs, ex6_sol, ex6_lhs.

# TODO: your code here

In [ ]:
# Check Exercise 6 -- run after your attempt.
from lib.grading import check

with check("Exercise 6"):
    coeffs = np.asarray(ex6_coeffs)
    rhs = np.asarray(ex6_rhs)
    sol = np.asarray(ex6_sol)
    lhs = np.asarray(ex6_lhs)
    assert coeffs.shape == (3, 3), (
        "three equations in three unknowns make a 3-by-3 grid -- one row per "
        "equation, one column per unknown"
    )
    assert rhs.shape == (3,) and sol.shape == (3,) and lhs.shape == (3,), (
        "the constants, the solution and the substituted values are each flat 3-entry "
        "arrays -- one number per equation or per unknown"
    )
    assert np.allclose(coeffs, np.array([[1, 1, 1], [2, -1, 1], [1, 2, -1]])), (
        "read the coefficients straight off the three equations, minus signs included "
        "-- the second row is 2, -1, 1 and the third is 1, 2, -1, columns in the "
        "order x, y, z"
    )
    assert np.allclose(rhs, np.array([6, 3, 2])), (
        "the constant terms are 6, 3 and 2 -- the numbers the three equations equal, "
        "in the order the equations are written"
    )
    assert np.allclose(lhs, np.sum(coeffs * sol, axis=1)), (
        "each entry of ex6_lhs should be one row of your grid multiplied entry by "
        "entry against your solution, then summed"
    )
    assert np.allclose(lhs, np.asarray(ex6_rhs)), (
        "substituting a solution back has to reproduce the constant terms -- if it "
        "does not, the values are not a solution of this system"
    )

### Solutions

In [ ]:
# --- Exercise 1 ---
ex1_linear = [True, False, True, False, False, True]
print("linear? (a)-(f):", ex1_linear)   # (b), (d) and (e) each break the rule

# --- Exercise 2 ---
# the first equation already gives y, so 2x + (3x - 1) = 9  ->  5x = 10
ex2_x = 2
ex2_y = 3 * ex2_x - 1
print("x =", ex2_x, " y =", ex2_y)      # 2, 5

# --- Exercise 3 ---
# the y terms are +3y and -3y, so adding the equations gives 6x = 24
ex3_x = 4
ex3_y = (25 - 4 * ex3_x) / 3
print("x =", ex3_x, " y =", ex3_y)      # 4, 3.0

# --- Exercise 4 ---
ex4_no_solution = "P"     # doubling x + 2y = 5 asks for 10, but the second says 7
ex4_infinite = "Q"        # doubling 3x - y = 4 IS the second equation
print("no solution:", ex4_no_solution, " infinitely many:", ex4_infinite)

# --- Exercise 5 ---
# first + third: 2x + 3y = 8;  second + third: 3x + y = 5
# tripling the second of those gives 9x + 3y = 15; subtracting 2x + 3y = 8
# leaves 7x = 7, so x = 1 and then y = 2
ex5_x, ex5_y = 1, 2
ex5_z = 6 - ex5_x - ex5_y
print("x, y, z =", ex5_x, ex5_y, ex5_z)  # 1, 2, 3

# --- Exercise 6 ---
ex6_coeffs = np.array([
    [1,  1,  1],
    [2, -1,  1],
    [1,  2, -1],
])
ex6_rhs = np.array([6, 3, 2])
ex6_sol = np.array([ex5_x, ex5_y, ex5_z])
ex6_lhs = np.sum(ex6_coeffs * ex6_sol, axis=1)
print("substituted:", ex6_lhs, " constants:", ex6_rhs)

## 10. Where this shows up in quantum

Measure a single qubit and exactly one of two things happens: you read a 0, or you
read a 1. Call the probabilities of those two outcomes $p_0$ and $p_1$. Because one
of them must occur,

$$
p_0 + p_1 = 1
$$

That is a linear equation. Both unknowns stand alone, each multiplied by a
coefficient of 1 and added -- exactly the shape of Section 1. It is the constraint
every legal qubit state obeys, and a state whose two probabilities sum to anything
else is not a state at all. That is why "print the norm first" is the standard
debugging move in every quantum notebook you will ever write.

One equation in two unknowns pins nothing down, which is where Section 3 began. So a
second measurement is needed. Quantum hardware reports a quantity written
$\langle Z \rangle$, which for a single qubit is exactly $p_0 - p_1$ -- how much the
0 outcome outweighs the 1 outcome. Now there are two equations,

$$
\begin{aligned}
p_0 + p_1 &= 1 \\
p_0 - p_1 &= \langle Z \rangle
\end{aligned}
$$

and adding them eliminates $p_1$, which is Section 4 with nothing changed but the
letters. The probabilities come straight back out.

Every measurement statistic the rest of this curriculum computes -- expectation
values, the energy of a molecule, the cost function a variational algorithm
minimizes -- is a linear expression in numbers just like these, and recovering them
from measurements is a linear system exactly like the one you just solved. For `n`
qubits there are $2^n$ outcomes rather than two, so the sum-to-one constraint becomes
one linear equation in $2^n$ unknowns. The systems get enormous. They never stop
being linear.

## Summary

- An equation is **linear** when every unknown appears alone, multiplied by a plain
  number and raised to the first power. A power, a product of two unknowns, a
  function, or a denominator breaks it.
- A **system** is several equations that must hold at once. **Substitution** works
  when one equation is already solved for an unknown; **elimination** works always,
  by scaling equations until an unknown cancels.
- **Substituting the answer back is the check.** It is the only one that does not
  depend on trusting the method you just used.
- Two equations in two unknowns are two lines, so there are exactly three outcomes:
  they cross (one solution), they are parallel (none), or they coincide (infinitely
  many).
- Strip the unknowns off a system and a **matrix** of coefficients is what remains,
  one row per equation and one column per unknown. Substituting a solution back is a
  row of that matrix multiplied against a column of values and summed -- the operation
  notebook 03 is built on.

**You finished notebook 1.** Move on to
[`02-matrices-add-subtract.ipynb`](02-matrices-add-subtract.ipynb), which gives that
grid a name, a shape, and its first two operations.